# Vision-Language Models — AI Engineering Interview Prep

Multimodal concepts and Claude vision API usage.

**Prerequisites**: Set `ANTHROPIC_API_KEY` environment variable.

In [ ]:
import anthropic
import os
import base64
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import io

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
MODEL = "claude-haiku-4-5-20251001"
print(f"Model: {MODEL}")

## 1. Vision-Language Model Architecture Overview

In [ ]:
# Architecture concepts — no API call needed
architecture_summary = """
Vision-Language Model (VLM) Architecture:

Image → [Vision Encoder] → Image Tokens → [Projection Layer] ─┐
                                                                 ├→ [LLM] → Text
Text  → [Text Tokenizer] → Text Tokens  ──────────────────────┘

Key Components:
1. Vision Encoder: Encodes image patches into embeddings
   - CLIP ViT (used by GPT-4V, LLaVA)
   - Custom CNN/ViT (used by Gemini, Claude)

2. Projection Layer: Maps vision features to LLM token space
   - MLP projection (LLaVA)
   - Q-Former (BLIP-2)
   - Linear projection

3. Large Language Model: Processes text + image tokens together
   - Treats image tokens like text tokens after projection
   - Causal attention over combined sequence

Common VLMs (2024-2025):
- Claude 3.x (Anthropic) — multimodal natively
- GPT-4o (OpenAI)
- Gemini 1.5 (Google)
- LLaVA-1.6 (open-source)
- Phi-3-vision (Microsoft)
"""
print(architecture_summary)

## 2. Generate a Test Image (No Download Required)

In [ ]:
# Create a synthetic chart image to analyze
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
models = ['Linear', 'Random\nForest', 'XGBoost', 'Neural\nNet']
accuracies = [0.78, 0.89, 0.92, 0.91]
axes[0].bar(models, accuracies, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'])
axes[0].set_ylim(0.7, 1.0)
axes[0].set_ylabel('Test Accuracy')
axes[0].set_title('Model Comparison')
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.002, f'{v:.2f}', ha='center')

# Learning curve
epochs = range(1, 21)
train_loss = [0.9*0.85**e for e in epochs]
val_loss   = [0.9*0.87**e + 0.02 for e in epochs]
axes[1].plot(epochs, train_loss, 'b-', label='Train Loss')
axes[1].plot(epochs, val_loss, 'r--', label='Val Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].set_title('Training Curves'); axes[1].legend()

plt.tight_layout()

# Save to bytes
buf = io.BytesIO()
plt.savefig(buf, format='png', dpi=100, bbox_inches='tight')
plt.close()
buf.seek(0)
image_bytes = buf.read()
image_b64 = base64.b64encode(image_bytes).decode('utf-8')

print(f"Generated chart image: {len(image_bytes)} bytes, b64: {len(image_b64)} chars")

## 3. Analyze Image with Claude Vision API

In [ ]:
# Send image to Claude for analysis
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": image_b64,
                    }
                },
                {
                    "type": "text",
                    "text": """Analyze this ML results chart. Please:
1. Describe what each subplot shows
2. Identify the best performing model and by how much
3. Comment on whether the training curves suggest overfitting
4. Give one recommendation based on these results"""
                }
            ]
        }
    ]
)

print("Claude's analysis:")
print("-" * 50)
print(message.content[0].text)
print(f"\nTokens used: {message.usage.input_tokens} input, {message.usage.output_tokens} output")

## 4. Multimodal Use Cases

In [ ]:
use_cases = {
    "Document Analysis": [
        "Extract table data from scanned invoices",
        "Parse structured forms and PDFs",
        "Convert handwritten notes to text",
    ],
    "Visual QA": [
        "Answer questions about scientific diagrams",
        "Interpret data visualizations",
        "Describe image content for accessibility",
    ],
    "Code + Screenshot": [
        "Debug UI from screenshots",
        "Generate code to recreate a design",
        "Identify issues in rendered output",
    ],
    "Medical/Scientific": [
        "Analyze medical imaging (with disclaimers)",
        "Count cells in microscopy images",
        "Detect anomalies in sensor readings",
    ]
}

for category, cases in use_cases.items():
    print(f"\n{category}:")
    for case in cases:
        print(f"  • {case}")

## Interview Tips

- **Image input**: Claude accepts base64 or URL. Max image size varies by model — check documentation.
- **Token cost**: images are encoded as a fixed number of tokens. 1 image ≈ 1500-4000 tokens depending on resolution.
- **CLIP intuition**: contrastively trained on (image, caption) pairs. Zero-shot classification via similarity.
- **Grounding vs pure description**: VLMs can ground text to image regions (bounding boxes) with some models.
- **Vision encoder choice**: ViT-L/14 (CLIP) vs SigLIP (Google) — SigLIP performs better on dense tasks.
- **Key papers**: CLIP (Radford 2021), BLIP-2 (Li 2023), LLaVA (Liu 2023), Flamingo (DeepMind 2022).